In [1]:
import numpy as np
import pandas as pd
import pulp
import matplotlib.pyplot as plt

In [3]:
prod_lines = ["A", "B", "C"]
metal_sheets = ["0.4 mm", "0.5 mm", "0.6 mm", "0.7 mm"]
demand = [500, 1200, 1500, 300]

manufacturing_costs = [
    [60, 50, 50, 45],
    [80, 70, 75, 70],
    [np.nan, 60, 60, np.nan],
]
manufacturing_time = [
    [0.4, 0.3, 0.3, 0.25],
    [0.8, 0.6, 0.75, 0.6],
    [np.nan, 0.5, 0.4, np.nan],
]
capacity = [500, 480, 370]

In [4]:
def solve_Chappie(prod_lines, metal_sheets, demand, manufacturing_costs, manufacturing_time, capacity):
    manufacturing_costs = np.asarray(manufacturing_costs, dtype=float)
    manufacturing_time = np.asarray(manufacturing_time, dtype=float)

    model = pulp.LpProblem("Chappie_Production_Optimization", pulp.LpMinimize)

    x = {
        (l, s): model.add_variable(f"x_{l}_{s}", lowBound=0)
        for i, l in enumerate(prod_lines)
        for j, s in enumerate(metal_sheets)
        if not np.isnan(manufacturing_costs[i, j])
    }

    model += pulp.lpSum(manufacturing_costs[prod_lines.index(l), metal_sheets.index(s)] * x[l, s] for (l, s) in x)

    # constraints
    for j, s in enumerate(metal_sheets):
        model += pulp.lpSum(x[l, s] for l in prod_lines if (l, s) in x) >= demand[j], f"Demand_{s}"

    for i, l in enumerate(prod_lines):
        model += pulp.lpSum(manufacturing_time[i, j] * x[l, s] for j, s in enumerate(metal_sheets) if (l, s) in x) <= capacity[i], f"Capacity_{l}"

    result = model.solve(pulp.HiGHS(msg=False))
    return model, x, result

In [5]:
model, x, result = solve_Chappie(prod_lines, metal_sheets, demand, manufacturing_costs, manufacturing_time, capacity)
print(result.status_str)
print(model)

Infeasible
Chappie_Production_Optimization:
MINIMIZE
60.0*x_A_0.4_mm + 50.0*x_A_0.5_mm + 50.0*x_A_0.6_mm + 45.0*x_A_0.7_mm + 80.0*x_B_0.4_mm + 70.0*x_B_0.5_mm + 75.0*x_B_0.6_mm + 70.0*x_B_0.7_mm + 60.0*x_C_0.5_mm + 60.0*x_C_0.6_mm + 0.0
SUBJECT TO
Demand_0.4_mm: x_A_0.4_mm + x_B_0.4_mm >= 500

Demand_0.5_mm: x_A_0.5_mm + x_B_0.5_mm + x_C_0.5_mm >= 1200

Demand_0.6_mm: x_A_0.6_mm + x_B_0.6_mm + x_C_0.6_mm >= 1500

Demand_0.7_mm: x_A_0.7_mm + x_B_0.7_mm >= 300

Capacity_A: 0.4 x_A_0.4_mm + 0.3 x_A_0.5_mm + 0.3 x_A_0.6_mm + 0.25 x_A_0.7_mm
 <= 500

Capacity_B: 0.8 x_B_0.4_mm + 0.6 x_B_0.5_mm + 0.75 x_B_0.6_mm + 0.6 x_B_0.7_mm
 <= 480

Capacity_C: 0.5 x_C_0.5_mm + 0.4 x_C_0.6_mm <= 370

VARIABLES
x_A_0.4_mm Continuous
x_A_0.5_mm Continuous
x_A_0.6_mm Continuous
x_A_0.7_mm Continuous
x_B_0.4_mm Continuous
x_B_0.5_mm Continuous
x_B_0.6_mm Continuous
x_B_0.7_mm Continuous
x_C_0.5_mm Continuous
x_C_0.6_mm Continuous



In [6]:
def max_production_Chappie(prod_lines, metal_sheets, demand, manufacturing_costs, manufacturing_time, capacity):
    manufacturing_costs = np.asarray(manufacturing_costs, dtype=float)
    manufacturing_time = np.asarray(manufacturing_time, dtype=float)

    model = pulp.LpProblem("Chappie_Max_Production", pulp.LpMaximize)

    x = {
        (l, s): model.add_variable(f"x_{l}_{s}", lowBound=0)
        for i, l in enumerate(prod_lines)
        for j, s in enumerate(metal_sheets)
        if not np.isnan(manufacturing_costs[i, j])
    }

    # objective: total tons produced
    model += pulp.lpSum(x.values())

    # constraints: never produce more than the demand, and respect the line capacities
    for j, s in enumerate(metal_sheets):
        model += pulp.lpSum(x[l, s] for l in prod_lines if (l, s) in x) <= demand[j], f"Demand_{s}"

    for i, l in enumerate(prod_lines):
        model += pulp.lpSum(manufacturing_time[i, j] * x[l, s] for j, s in enumerate(metal_sheets) if (l, s) in x) <= capacity[i], f"Capacity_{l}"

    result = model.solve(pulp.HiGHS(msg=False))
    return model, x, result

In [7]:
model_max, x_max, result_max = max_production_Chappie(prod_lines, metal_sheets, demand, manufacturing_costs, manufacturing_time, capacity)
if result_max.status_str == "Optimal":
    plan = pd.DataFrame(0.0, index=prod_lines, columns=metal_sheets)
    for (l, s), var in x_max.items():
        plan.loc[l, s] = var.value()
    print("Production plan (tons):")
    print(plan)

    summary = pd.DataFrame({"demand": demand, "produced": plan.sum().values}, index=metal_sheets)
    summary["shortage"] = summary["demand"] - summary["produced"]
    print("\nDemand vs production:")
    print(summary)
    print("\nMaximum total production:", round(result_max.objective, 2), "tons of", sum(demand))

Production plan (tons):
   0.4 mm      0.5 mm  0.6 mm  0.7 mm
A    0.00  841.666667   575.0   300.0
B  331.25  358.333333     0.0     0.0
C    0.00    0.000000   925.0     0.0

Demand vs production:
        demand  produced  shortage
0.4 mm     500    331.25    168.75
0.5 mm    1200   1200.00      0.00
0.6 mm    1500   1500.00      0.00
0.7 mm     300    300.00      0.00

Maximum total production: 3331.25 tons of 3500
